In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime

from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

EMBED_PATH = ROOT / "data/embeddings/base_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = ROOT / "data/experiments" / EMBED_NAME / "hypersphere"

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [3]:
embeds = torch.load(EMBED_PATH, weights_only=False)
cls_tokens = embeds["cls_tokens"]

In [4]:
conn = sqlite3.connect(ROOT / "data/sql/metadata.db")

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [ ]:
%load_ext autoreload
%autoreload 2

from src.algorithims.hypersphere import CandidateCleaner, HypersphereEvaluator, HypersphereCover

cleaner = CandidateCleaner()
cover = HypersphereCover(cleaner=cleaner)

eval = HypersphereEvaluator()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    spheres, spheres_df = cover.run(cat_emb, output_dir=outputs_dir)

    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df = eval.overlaps(cat_emb, spheres)
    overlaps_df.to_csv(outputs_dir / "overlaps.csv", index=False)

    good_any, good_counts = eval.inside_any_count(good_test_emb, spheres)
    defect_any, defect_counts = eval.inside_any_count(defect_test_emb, spheres)

    auroc = eval.scores(good_test_emb, defect_test_emb, spheres)

    metadata = {
        "category": category,
        "K_frac": 0.05,
        "start_growth": 1.2,
        "min_growth": 1,
        "reg": 1e-4,
        "n_spheres": len(spheres),
        "auroc": auroc,
        "good_inside": int(good_any.sum()),
        "defect_inside": int(defect_any.sum())
    }

    with open(outputs_dir / "metadata.json", "w") as f:
        json.dump(metadata, f)



Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper
